In [ ]:
# (C) Martin Reißel

from sympy import *

from IPython.display import display, Math, Latex
from sympy.interactive import printing
printing.init_printing(use_latex='mathjax')
nlatex = lambda A: latex(A, mat_str='matrix', mat_delim='')
platex = lambda A: latex(A, mat_str='pmatrix', mat_delim='')

# Aufgabenstellung

Bestimmen Sie den natürlichen kubischen Spline durch die Punkte

In [ ]:
xi = Matrix([-1, 0, 1])
yi = Matrix([ 2, 1, 0])
display(Math('x_i :' + nlatex(xi.T) + r'\qquad y_i :' + nlatex(yi.T)))

# Lösung

* ein kubischer Spline setzt sich zusammen aus Polynomen vom Grad 3 auf den Intervallen $[x_i,x_{i+1}]$
* an den Nahtstellen $x_i$ müssen die Polynome zweimal stetig differenzierbar ineinander übergehen
* als Abschlussbedingung wird gefordert, dass die zweite Ableitung der Polynome an $x_0$ und $x_n$ verschwindet

* in unserem Fall haben wir 3 Datenpunkte, so dass der Spline sich aus zwei kubischen Polynomen
$p_l$ auf $[x_0,x_{1}]$ und $p_r$ auf $[x_1,x_{2}]$
zusammensetzt

In [ ]:
al,bl,cl,dl,ar,br,cr,dr,x = symbols('a_l,b_l,c_l,d_l,a_r,b_r,c_r,d_r,x', real = True)

pl = Lambda(x, al + bl * x + cl * x**2 + dl * x**3)
pr = Lambda(x, ar + br * x + cr * x**2 + dr * x**3)

display(Math('p_l(x) = ' + latex(pl(x))))
display(Math('p_r(x) = ' + latex(pr(x))))

* der Spline ist bestimmt, wenn wir die 8 Koeffizienten der beiden Polynome berechnet haben
* dazu benötigen wir 8 Gleichungen

* zunächst haben wir die Interpolationsbedingungen, d.h. der Spline
  soll bei $x_i$ die Werte $y_i$ annehmen, also
  

\begin{align*}
p_l(x_0) = y_0,
\quad
p_l(x_1) = y_1,
\quad
p_r(x_1) = y_1,
\quad
p_r(x_2) = y_2
\end{align*}

* setzen wir die Zahlen ein, so erhalten wir die ersten 4 linearen Gleichungen
für die Polynomkoeffizienten

In [ ]:
display(Math(r'{} = {}'.format(latex(pl(xi[0])), latex(yi[0]))))
display(Math(r'{} = {}'.format(latex(pl(xi[1])), latex(yi[1]))))
display(Math(r'{} = {}'.format(latex(pr(xi[1])), latex(yi[1]))))
display(Math(r'{} = {}'.format(latex(pr(xi[2])), latex(yi[2]))))

* wir haben eine Nahtstelle bei $x_1$
* zweifache stetige Differenzierbarkeit bei $x_1$ bedeutet


\begin{align*}
p_l'(x_1) = p_r'(x_1),
\quad
p_l''(x_1) = p_r''(x_1)
\end{align*}

* mit

In [ ]:
pls = Lambda(x, pl(x).diff(x))
plss = Lambda(x, pls(x).diff(x))

prs = Lambda(x, pr(x).diff(x))
prss = Lambda(x, prs(x).diff(x))

display(Math("p_l'(x) = {} \qquad p_l''(x) = {}".format(latex(pls(x)), latex(plss(x)))))
display(Math("p_r'(x) = {} \qquad p_r''(x) = {}".format(latex(prs(x)), latex(prss(x)))))

   erhalten wir 2 weitere lineare Gleichungen

In [ ]:
display(Math(r'{} = {}'.format(latex(pls(xi[1])) , latex(prs(xi[1])))))
display(Math(r'{} = {}'.format(latex(plss(xi[1])), latex(prss(xi[1])))))

- die Abschlussbedingungen

\begin{align*}
p_l''(x_0) = p_r''(x_2) = 0
\end{align*}
  liefern schließlich die letzten beiden linearen Gleichungen

In [ ]:
display(Math(r'{} = {}'.format(latex(plss(xi[0])), 0)))
display(Math(r'{} = {}'.format(latex(prss(xi[2])), 0)))

- insgesamt erhalten wir das folgende quadratische lineare Gleichungssystem

In [ ]:
xx = Matrix([al,bl,cl,dl,ar,br,cr,dr])

links = [
pl(xi[0]),
pl(xi[1]),
pr(xi[1]),
pr(xi[2]),
pls(xi[1])-prs(xi[1]),
plss(xi[1])-prss(xi[1]),
plss(xi[0]),
prss(xi[2])
]

A = zeros(xx.shape[0])
for j,xxx in enumerate(xx):
    for i,gg in enumerate(links):
        A[i,j] = gg.coeff(xxx)
        
b = Matrix([yi[0], yi[1], yi[1], yi[2], 0, 0, 0, 0])

display(Math(r'{} {} = {}'.format(platex(A), platex(xx), platex(b))))

In [ ]:
xs = solve(A*xx - b)
xs

In [ ]:
al,bl,cl,dl,ar,br,cr,dr = xx.subs(xs)

x = symbols('x', real = True)

pl = Lambda(x, al + bl * x + cl * x**2 + dl * x**3)
pr = Lambda(x, ar + br * x + cr * x**2 + dr * x**3)

s = Lambda(x, Piecewise((pl(x), x <= 0), (pr(x), x > 0)))
Math('s(x) = ' + latex(s(x)))

In [ ]:
%matplotlib inline
plot(s(x), (x, xi[0]-1, xi[-1] + 1), adaptive = False);